# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [29]:
#Part 1: First is I need to load the tools to read all the files I need to read.

import re          # imports pattern searching tools from lecture 8
import Bio.Seq     # imports DNA sequence tools from lecture 9
import Bio.SeqIO   # imports the BioSeq that reads the fastq file from lecture 9

seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))
print(f"Found {len(seqreads)} sequencing reads.") #I used lecture 9 for this 2 lines of code but had to use Claude to make sure mine is correct 



#Part 2: I need to save the sequences we are looking for. 
ha_end = "CCGGATTTGCATATAATGATGCACCAT"   # defining the end of HA from hw04 instructions
na_end = "CACGATAGATAAATAATAGTGCACCAT"   # defining the end of NA from hw04 instructions
upstream = "AGGCGGCCGC" #end tag

#Part 3: I need to make the search patterns from re.compile in lecture 8
#I'm using the defined gene ends + the tag
ha_pattern = re.compile(ha_end + upstream) 
na_pattern = re.compile(na_end + upstream)

#Part 4: Parsing functions
# I used the same structure as read_barcode from lecture 9 , this is to classify the reads as either HA or NA. 
def classify_read(seqread):
    """Decide if a read comes from HA or NA.

    Args:
        seqread: a read in the file's orientation (Seq object or string).

    Returns:
        "HA", "NA", or None if the read matches neither.
    """
    # I had to use Claude here on what to do after, but I realized there was a hint
    # in lecture 9 to flip it to the forward orientation and make it capital letters
    flipped = str(Bio.Seq.Seq(str(seqread)).reverse_complement()).upper()

    # then we check for HA then NA, I used if and elif from lecture 6
    if ha_pattern.search(flipped) is not None:
        return "HA"
    elif na_pattern.search(flipped) is not None:
        return "NA"
    else:
        return None

  #I ran the first part up just to make sure it is working, then it found 10000 sequencing reads


Found 10000 sequencing reads.


In [30]:
# Part 5: Test cases to make sure I am classifying reads correctly, mostly from lecture 9 but I had to use Claude to make sure I wrote the lines correctly
test_cases = [
    # positive: good HA read should give "HA"
    ("CCCCGGGGAAAATTTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGG", "HA", "positive: clean HA read"),
    # positive: good NA read should give "NA"
    ("CCCCGGGGAAAATTTTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTG", "NA", "positive: clean NA read"),
    # positive: lower case should still work because of .upper()
    ("ccccggggaaaattttgcggccgcctatggtgcatcattatatgcaaatccgg", "HA", "positive: lower case HA read"),
    # negative: last 4 letters changed, so it matches neither gene
    ("CCCCGGGGAAAATTTTGCGGCCGCCTATGGTGCATCATTATATGCAAATGGGG", None, "negative: end matches neither HA nor NA"),
    # negative: tag has one wrong letter (GCGGCCGCAT instead of GCGGCCGCCT)
    ("CCCCGGGGAAAATTTTGCGGCCGCATATGGTGCATCATTATATGCAAATCCGG", None, "negative: tag is wrong"),
]

print(run_tests(classify_read, test_cases))

# Part 6: run it on all 10,000 reads and count 
gene_counts = {"HA": 0, "NA": 0}   # I need to start both counts at 0
unclassified = []                  # This so that it keep the reads that match neither

for read in seqreads:
    gene = classify_read(read.seq)
    if gene is None:
        unclassified.append(read)   # save it and not throw it away
    else:
        gene_counts[gene] += 1      # add one to HA or NA

print("HA reads:", gene_counts["HA"])
print("NA reads:", gene_counts["NA"])
print("Reads matching neither:", len(unclassified))

PASS  positive: clean HA read
PASS  positive: clean NA read
PASS  positive: lower case HA read
PASS  negative: end matches neither HA nor NA
PASS  negative: tag is wrong
0 tests failed
HA reads: 5316
NA reads: 3973
Reads matching neither: 711


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [31]:
# To check if they have a valid barcode, I used read_barcode is from lecture 9 in step 3 the loop is like in step 4
# I used Claude to help explain this code

# function from lecture 09: gives back the barcode, or None if it is not valid
def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
    # flip the read and make it capital letters 
    flipped = str(Bio.Seq.Seq(str(seqread)).reverse_complement()).upper()

    # pattern: the tag, then exactly 16 real bases, then the end of the read
    barcode_pattern = re.compile(upstream + r"(?P<barcode>[ACGTN]{" + str(bclen) + r"})$") #VS Code helped me complete this line

    # look for it
    match = barcode_pattern.search(flipped)
    if match is None:
        return None                     # no valid barcode
    return match.group("barcode")       # the barcode


# define and start both counts at 0
ha_invalid = 0
na_invalid = 0

# go through every read
for read in seqreads:
    gene = classify_read(read.seq)      # "HA", "NA", or None (from Q1)
    barcode = read_barcode(read.seq)    # barcode, or None if bad

    # only count reads where the barcode is bad
    if barcode is None:
        if gene == "HA":
            ha_invalid += 1
        elif gene == "NA":
            na_invalid += 1

print("HA reads with no valid barcode:", ha_invalid)
print("NA reads with no valid barcode:", na_invalid)

HA reads with no valid barcode: 70
NA reads with no valid barcode: 63


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [32]:
# So I need to find the HA barcode with the most counts, and the NA barcode with the most counts
# The way I do it is from the counting with a dictionary is from lecture 8 (warm up) and lecture 9 (Step 4),
# for loops and if statements are from lecture 6
# I used Claude to help me reorganize the code and explain it to me, but I had to use my own logic to make it work.

# First I need to count every barcode, separately for HA and NA 
ha_counts = {}   # empty tally sheet for HA
na_counts = {}   # empty tally sheet for NA

for read in seqreads:
    gene = classify_read(read.seq)        # "HA", "NA", or None (from Q1)
    barcode = read_barcode(read.seq)      # the barcode, or None (from Q2)

    if barcode is not None:               # only count valid barcodes
        if gene == "HA":
            if barcode in ha_counts:
                ha_counts[barcode] += 1   # if it is seen before, add 1
            else:
                ha_counts[barcode] = 1    # if it is seen for the first time, start at 1

        elif gene == "NA":
            if barcode in na_counts:
                na_counts[barcode] += 1
            else:
                na_counts[barcode] = 1

# Now finding the biggest count for HA
best_ha_barcode = ""   # defining it first 
best_ha_count = 0      # biggest count so far

for barcode in ha_counts:
    if ha_counts[barcode] > best_ha_count:     # I am defining to see if it is bigger than the winner 
        best_ha_barcode = barcode              # then this is the new winner
        best_ha_count = ha_counts[barcode]

print("Top HA barcode:", best_ha_barcode, "with", best_ha_count, "counts")


# Same thing for NA too
best_na_barcode = ""
best_na_count = 0

for barcode in na_counts:
    if na_counts[barcode] > best_na_count:
        best_na_barcode = barcode
        best_na_count = na_counts[barcode]

print("Top NA barcode:", best_na_barcode, "with", best_na_count, "counts")


Top HA barcode: CCCGACCCGACATTAA with 155 counts
Top NA barcode: ACCAGTTCTCCCCGGG with 152 counts
